In [ ]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic

In [ ]:
%pip install litellm
%pip install -U crewai

In [ ]:
import os
import requests
import litellm
from crewai.llm import LLM
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool
from dotenv import load_dotenv

load_dotenv()

# Configure LiteLLM
litellm.drop_params = True
# Monkey patch litellm.completion to handle parameter mapping
original_completion = litellm.completion

def patched_completion(*args, **kwargs):
    # If max_tokens is present and max_completion_tokens is not, map it
    if 'max_tokens' in kwargs and 'max_completion_tokens' not in kwargs:
        kwargs['max_completion_tokens'] = kwargs.pop('max_tokens')
    return original_completion(*args, **kwargs)

# Apply the patch
litellm.completion = patched_completion

In [ ]:
# Tavily API Key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [ ]:
# ---- Custom CrewAI Tool for Web Search ----
# Same as the demo 1 -> this version is hardened to handle errors and timeouts gracefully, and to return a more user-friendly message if no results are found.
# 
class TavilySearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"
        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload, timeout=30) 
        response.raise_for_status()
        data = response.json()

        results = []
        for r in data.get("results", []):
            title = r.get("title", "No title")
            link = r.get("url", "No URL")
            results.append(f"{title} - {link}")

        return "\n".join(results) if results else "No web results found."


search_tool = TavilySearchTool()

# ---- Azure LLM - FIXED ----
# Using max_tokens (not max_completion_tokens) with the monkey patch
llm = LLM(
    model=f"azure/{os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')}",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    is_litellm=True,
    temperature=1, # some models don't support temperature
    max_tokens=3500  # This will be converted to max_completion_tokens
)

In [ ]:
# --------------------------------------------------
# Agents
# --------------------------------------------------
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for healthcare",
    backstory=(
        "You are an expert in artificial intelligence and stay updated "
        "with the latest research trends in healthcare."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=2, # gives the researcher one extra reasoning/tool-call loop before it must finalize an answer.
    tools=[search_tool]
)

writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory=(
        "You are an experienced technical writer with expertise in "
        "summarizing healthcare research for executives."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [ ]:
# --------------------------------------------------
# SERIAL EXECUTION
# --------------------------------------------------
task_research = Task(
    description=(
        "Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4 sentences."
        "Do not call tools again after getting results."
    ),
    expected_output=(
        "Detailed notes on three advancements, with names and explanations."
    ),
    agent=researcher
)

task_write = Task(
    description=(
        "Write a short executive summary using the research notes provided by the AI Researcher. "
        "Limit the answer to about 100 words."
    ),
    expected_output=(
        "An executive summary report of the top 3 AI advancements in healthcare."
    ),
    agent=writer,
    context=[task_research]
)

print("\n=== SERIAL EXECUTION ===")

crew_serial = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

serial_result = await crew_serial.kickoff_async()

print("\n[Serial Result]:\n")
try:
    print(serial_result.raw)
except AttributeError:
    print(serial_result)

In [ ]:
# --------------------------------------------------
# PARALLEL EXECUTION
# --------------------------------------------------
# 1. async_execution=True on task_parallel_1 tells CrewAI not to block the main execution thread waiting for this task to finish before starting the next one.
# 2. Because task_parallel_2 has no context= argument, it doesn't depend on task_parallel_1's output — the two tasks are independent, so CrewAI kicks both off essentially at once rather than waiting for one to finish before starting the other.
# 3. A brand-new Crew (crew_parallel) is assembled with these two independent tasks and run separately from the serial crew — this notebook literally runs two Crews in one session to show the contrast side by side.
task_parallel_1 = Task(
    description=(
        "Use web search to list 5 AI companies focusing on drug discovery. "
        "For each company, give one short line about what they specialize in."
    ),
    expected_output="Company names and what they specialize in.",
    async_execution=True,
    agent=researcher
)

task_parallel_2 = Task(
    description=(
        "Write a short report on how AI is transforming patient diagnostics. "
        "Limit the answer to about 100 words."
    ),
    expected_output="A short report with examples and explanation.",
    agent=writer
)

print("\n=== PARALLEL EXECUTION ===")

crew_parallel = Crew(
    agents=[researcher, writer],
    tasks=[task_parallel_1, task_parallel_2],
    verbose=True
)

# This line kicks off the crew, lets task_parallel_1 (async) run concurrently rather than blocking, 
# Waits for the whole crew's work to wrap up, and stores the final combined result in parallel_result.
parallel_result = await crew_parallel.kickoff_async() # 

print("\n[Parallel Result]:\n")
try:
    print(parallel_result.raw)
except AttributeError:
    print(parallel_result)